# OSINT social diagram

In this notebook we will:
- First, extract the list of events in `photos` directory 
- for each event, we will extract the faces and put them in the `faces` directory
- Then, we will group the faces by similarity and create clusters in `grouped_faces` directory
- We will get the images in `photos/profile_pics` and find the closest cluster for each profile picture
- We will then create a social diagram based on the following rules:
- - If two people appear in the same event, their connection gains 1 point, applied once per event.
- - If two people appear in the same image, their connection gains 2 points if there are 4 or more people in the image, otherwise it gains 5 points.

# Assumptions:

- The `photos` directory contains subdirectories for each event, and each subdirectory contains images of that event.
- The `faces` directory will store the extracted faces from the event images.
- The `grouped_faces` directory will store the clustered faces.
- The `photos/profile_pics` directory contains profile pictures of individuals. Each profile picture name corresponds to the name of the individual.
- The social diagram will be represented as an  graph where nodes represent individuals and edges represent connections between them, with weights based on the rules defined above.
- Individuals are 18-25 years old, from various ethnic backgrounds.
- The number of people in an image is determined by the number of faces detected by the face detection algorithm.
- Unidentified faces will be labeled as "Unknown" and will not be included in the social diagram.

# Graph rules:
- If two people appear in the same event, their connection gains 1 point, applied once
- If two people appear in the same image, their connection gains 2 points if there are 4 or more people in the image, otherwise it gains 5 points. Applied once per image. For simplicity, we will assume all images are distinct and do not contain duplicates.

We will use insightface library because its face detection and recognition models fit our population. The models are trained on a diverse dataset and have been shown to perform well across different ethnicities and age groups. The models are also robust to variations in lighting, pose, and occlusions, which are common in real-world scenarios.

We will avoid using face_recognition library with dlib because of its low performance on non-Caucasian faces, which is a concern given our population. 

#### Step 1: Intalling and importing required libraries

In [2]:
!pip install numpy scikit-learn tqdm insightface onnxruntime opencv-python networkx pyvis matplotlib

In [3]:
import os
from tqdm import tqdm
import cv2
from insightface.app import FaceAnalysis
import numpy as np
from sklearn.cluster import DBSCAN
import shutil
import networkx as nx
import pyvis
import matplotlib.pyplot as plt

app = FaceAnalysis(name="buffalo_l")
app.prepare(ctx_id=0)

d:\Projects\osint\Minerva\.venv\Lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:147: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\Me/.insightface\models\buffalo_l\1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\Me/.insightface\models\buffalo_l\2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\Me/.insightface\models\buffalo_l\det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\Me/.insightface\models\buffalo_l\genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\Me/.insightface\models\buffalo_l\w600k_r50.onnx recognition ['None', 3, 112, 112] 127.5 127.5
set det

#### Step 1.1: create and check directories


In [4]:
os.makedirs('photos/profile_pics', exist_ok=True) # in case profile_pics does not exist

#### Step 2.1: Extracting event names from the `photos` directory

In [5]:
# First, lets get the  list of events and images
events = os.listdir('photos')
images = []
for event in events:
    if event == 'profile_pics':
        continue
    path = f'photos/{event}'
    event_imgs = os.listdir(path)
    for img in event_imgs:
        images.append(f'{path}/{img}')
    print(f"{event}\t-\t{len(event_imgs)} images")

print(f'Found {len(images)} images in total.')

2024-03-15__Quinquatria	-	24 images
2025-03-21__Quinquatria	-	37 images
Claude_17.10	-	21 images
Found 82 images in total.


#### Step 2.2: Extract faces from `images` in each event and save them in the `faces` directory

In [ ]:
FACE_DIR = "faces"
os.makedirs(FACE_DIR, exist_ok=True)
embeddings = []
face_files = []
event_name = []
img_names  = []
face_count = {}

for image in tqdm(images):
    img = cv2.imdecode(
        np.fromfile(f"{image}", dtype=np.uint8),
        cv2.IMREAD_COLOR
    )
    event,image_name = os.path.split(image)
    
    faces = app.get(img)
    face_count[image] = len(faces)
    for i in range(len(faces)):
        h, w = img.shape[:2]

        x1, y1, x2, y2 = faces[i].bbox.astype(int)
        embedding = faces[i].embedding
        embedding = embedding / np.linalg.norm(embedding)
        embeddings.append(faces[i].embedding)
        event_name.append(event)
        img_names.append(image_name)

        x1 = max(0, x1)
        y1 = max(0, y1)
        x2 = min(w, x2)
        y2 = min(h, y2)

        if x2 > x1 and y2 > y1:
            crop = img[y1:y2, x1:x2]
            face_files.append(f"{image_name}_face_{i}.png")
            cv2.imwrite(f"{FACE_DIR}/{image_name}_face_{i}.png", crop)
        else:
            face_files.append(None)
            print("Invalid crop")


  7%|▋         | 6/82 [00:04<00:55,  1.36it/s]d:\Projects\osint\Minerva\.venv\Lib\site-packages\insightface\utils\face_align.py:23: FutureWarning: `estimate` is deprecated since version 0.26 and will be removed in version 2.2. Please use `SimilarityTransform.from_estimate` class constructor instead.
  tform.estimate(lmk, dst)
100%|██████████| 82/82 [06:37<00:00,  4.85s/it]


Make sure everything is a numpy array.

In [7]:
embeddings = np.array(embeddings)
face_files = np.array(face_files)
event_name = np.array(event_name)
img_names  = np.array(img_names)

#### Step 2.3: Get embeddings of the profile pictures

In [8]:
PROFILES = f'photos/profile_pics'
profile_embedding = []
names = []
for profile in tqdm(os.listdir(PROFILES)):
    img = cv2.imdecode(
        np.fromfile(f"{PROFILES}/{profile}", dtype=np.uint8),
        cv2.IMREAD_COLOR
    )
    faces = app.get(img)
    name = os.path.basename(profile)

    if len(faces) == 1:
        face = faces[0]
        embedding = face.embedding
        embedding = embedding / np.linalg.norm(embedding)
        profile_embedding.append(embedding)
        names.append(name)
    elif len(faces) > 1:
        # get the largest face ine image as a proxy for focus
        face = max(
            faces,
            key=lambda f: (f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1])
            )

        embedding = face.embedding
        embedding = embedding / np.linalg.norm(embedding)
        profile_embedding.append(embedding)
        names.append(name)



100%|██████████| 200/200 [14:13<00:00,  4.27s/it]


#### Step 3: Group the faces by similarity and create clusters in `grouped_faces` directory 

In [ ]:
GROUPED_DIR = 'grouped_faces'
os.makedirs(GROUPED_DIR, exist_ok=True)

embeddings = np.array(embeddings)

db = DBSCAN(
    eps=0.45,
    min_samples=2,
    metric="cosine"
)
labels = db.fit_predict(embeddings)

for i in tqdm(range(len(labels))):
    l = labels[i]
    person_id = "unknown"
    if l != -1:
        person_id = f'person_{l}'
    if not face_files[i]:
        continue

    os.makedirs(f'{GROUPED_DIR}/{person_id}',exist_ok=True)
    from_path = f'{FACE_DIR}/{face_files[i]}'
    to_path = f'{GROUPED_DIR}/{person_id}'
    shutil.copy2(from_path, to_path)

    


100%|██████████| 104/104 [00:02<00:00, 35.67it/s]


#### Step 3.1: calculate the cluster centers

In [10]:
cluster_centers = []
cluster_labels = []

for l in tqdm(set(labels)):
    if l == -1:
        continue
    avg_embedding = np.mean(embeddings[labels == l], axis=0)
    # normalization
    avg_embedding = avg_embedding / np.linalg.norm(avg_embedding)
    cluster_centers.append(avg_embedding)
    cluster_labels.append(l)

cluster_centers = np.array(cluster_centers)

100%|██████████| 21/21 [00:00<00:00, 12107.27it/s]


In [11]:
def find_profile(center, profiles, names, threshold=0.55):
    if len(profiles) < 1:
        return None
    best_similarity = -10
    cos_similarity = profiles @ center
    best_idx = np.argmax(cos_similarity)
    best_similarity = cos_similarity[best_idx]
    
    if best_similarity < 1 - threshold:
        return None
    
    return names[best_idx]



#### Step 3.2: Rename the clusters based on the closest profile picture

Because several clusters may be closest to the same person, we will match each cluster to the closest profile, and if it exists, we merge these directories. This way, we will have a single directory for each person, containing all the faces that belong to them.

If we do it the other way, matching each profile to the closest cluster, we may end up with incorrectly named clusters because we are forcing a match to a profile picture even if the cluster does not belong to that person.

In [12]:
# we need to map each cluster label to a profile name
profiles = []

for cluster_idx in tqdm(range(len(cluster_centers))):
    center_embedding = cluster_centers[cluster_idx]
    center_label     = cluster_labels[cluster_idx]
    best_profile     = find_profile(center_embedding, profile_embedding, names)
    if not best_profile:
        profiles.append(f'person_{center_label}')
        continue
    
    best_profile     = os.path.splitext(best_profile)[0]

    profiles.append(best_profile)
    # now we need to rename the best cluster label to name
    original_dir = f'grouped_faces/person_{center_label}'
    new_dir      = f'grouped_faces/{best_profile}'

    if not os.path.exists(original_dir):
        continue

    if not os.path.exists(new_dir):
        os.rename(original_dir, new_dir)
    else:
        # if a cluster had that name, then merge them
        shutil.copytree(original_dir, new_dir, dirs_exist_ok=True)
        shutil.rmtree(original_dir)




100%|██████████| 20/20 [00:00<00:00, 761.37it/s]


# Summary of variables we have until now:
- `embeddings` - a list of face embeddings for each face in the `faces` directory
- `labels` - a list of labels corresponding to each face embedding
- `cluster_labels` - a list of cluster labels corresponding to each face embedding
- `event_names` - a list of event names corresponding to each face embedding
- `cluster_centers` - a list of cluster centers for each cluster in the `grouped_faces` directory
- `profile_embeddings` - a list of embeddings for each profile picture in the `photos/profile_pics` directory
- `profile_names` - a list of names corresponding to each profile picture in the `photos/profile_pics` directory
- `profiles` - a list of profiles for each cluster

At this point we have already extracted the faces from the images, and corresponded each face with a profile picture.

before we make the social diagram, we will create a mapping of the following:
- Each event to the list of people who attended that event
- Each image to the list of people who appeared in that image
- a matrix of connections between each pair of people

In [13]:
events = {}
images = {}
connections_matrix = {}

for i in tqdm(range(len(face_files))):
    event = os.path.split(event_name[i])[1]
    image_name = img_names[i]
    label = labels[i]

    if label == -1:
        continue

    cluster_idx = cluster_labels.index(label)
    profile = profiles[cluster_idx]

    if event not in events:
        events[event]      = set()
    if image_name not in images:
        images[image_name] = set()

    events[event].add(profile)
    images[image_name].add(profile)
    

100%|██████████| 104/104 [00:00<00:00, 281787.87it/s]


#### Point Calculation

In [14]:
for img in face_count:
    key = os.path.split(img)[1]
    if key not in images:
        continue
    for person_1st in images[key]:
        if person_1st not in connections_matrix:
            connections_matrix[person_1st] = {}

        for person_2nd in images[key]:
            if person_2nd == person_1st:
                continue
            points = 0
            if face_count[img] <= 3:
                points = 5
            else:
                points = 2
            connections_matrix[person_1st][person_2nd] = connections_matrix[person_1st].get(person_2nd, 0) + points


In [15]:
for event in events:
    for person_1st in events[event]:
        if person_1st not in connections_matrix:
            connections_matrix[person_1st] = {}

        for person_2nd in events[event]:
            if person_2nd == person_1st:
                continue
            connections_matrix[person_1st][person_2nd] = connections_matrix[person_1st].get(person_2nd, 0) + 1


#### Normalization by dividing by global maximum

In [16]:
global_max = 1
for p1 in connections_matrix:
    for p2 in connections_matrix[p1]:
        global_max = max(global_max, connections_matrix[p1][p2])
for p1 in connections_matrix:
    for p2 in connections_matrix[p1]:
        connections_matrix[p1][p2] = connections_matrix[p1][p2] / global_max


In [17]:
def get_image(name):
    directory = f"grouped_faces/{name}"
    if not os.path.exists(directory):
        return None

    return os.path.join(directory, os.listdir(directory)[0])

# Graph Creation

In [18]:
from pyvis.network import Network

N = Network(height="800px", width="100%", notebook=True)

for person_1, connections in connections_matrix.items():
    person1_img = get_image(person_1)

    if person1_img and person_1 not in N.get_nodes():
        N.add_node(
            person_1,
            label=os.path.splitext(person_1)[0],
            image=person1_img,
            shape="image"
        )

    for person_2, weight in connections.items():

        person2_img = get_image(person_2)

        if person2_img and person_2 not in N.get_nodes():
                N.add_node(
                    person_2,
                    label=os.path.splitext(person_2)[0],
                    image=person2_img,
                    shape="image"
                )

        N.add_edge(person_1, person_2, value=weight)

N.show("people_graph.html")

plt.show()

people_graph.html
